# CenpAI dataset v2 (Phase 5, steps 1 and 2)

Downloads every source in `ml/datasets.yaml`, checks it, and builds dataset **v2** with `ml.dataset`:
images grouped by their original source name, near duplicates merged, PH BB v5 split 70/15/15 by group,
Open Images kept on Google's official splits, every other internet source used for training only.
The last cells print the two leak counts, which must both be `0`.

Rules (`docs/protocols/data_and_phone_test.md`):
- Only the licensed sources listed in `docs/datasets.md`. No images from a web search or social media.
- The family-car test videos are never read by this notebook or any other training notebook.
- No keys in the notebook. Every number it prints is measured in this run.

**Before running** (Colab, any runtime; a GPU is only used by the optional keyboard review cell):
add the secret `ROBOFLOW_API_KEY` (key icon on the left, switch on *Notebook access*).

Storage: raw downloads are zipped once to `MyDrive/CenpAI/data/raw-zips/` and restored from there on later runs.
Work happens on the local disk (`/content/work`), because reading thousands of small files through the Drive mount is slow.

In [ ]:
# 1. Setup
!pip -q install roboflow fiftyone pyyaml gdown
import os, sys, json, shutil, zipfile, random, subprocess, hashlib
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")

REPO_BRANCH = "develop"  #@param {type:"string"}
REPO = Path("/content/cenpai")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, "https://github.com/RedGla/cenpai", str(REPO)], check=True)
sys.path.insert(0, str(REPO))
from ml import dataset as ds
import yaml

ROOT = Path("/content/drive/MyDrive/CenpAI")
ZIPS = ROOT / "data" / "raw-zips"
OUT_DRIVE = ROOT / "data" / "processed"
WORK = Path("/content/work")
RAW = WORK / "raw"
for p in (ZIPS, OUT_DRIVE, RAW):
    p.mkdir(parents=True, exist_ok=True)
os.environ[ds.DATA_ENV] = str(WORK)          # ml/datasets.yaml paths are relative to this folder
CFG = yaml.safe_load((REPO / "ml" / "datasets.yaml").read_text())
print("dataset version", CFG["version"], "| sources:", [s["name"] for s in CFG["sources"]])

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None
print("ROBOFLOW_API_KEY set:", bool(secret("ROBOFLOW_API_KEY")))

In [ ]:
# 2. Helper: restore a raw source from its zip on Drive, or download it once and zip it to Drive
def src_dir(name):
    return ds.resolve_path(next(s["path"] for s in CFG["sources"] if s["name"] == name))

def fetch(name, download, zip_name=None):
    dest = src_dir(name)
    z = ZIPS / (zip_name or f"{dest.name}.zip")
    if dest.is_dir() and any(dest.iterdir()):
        print(f"{name}: already on local disk")
    elif z.is_file():
        shutil.copy(z, "/content/tmp.zip")
        with zipfile.ZipFile("/content/tmp.zip") as f:
            f.extractall(dest)
        print(f"{name}: restored from {z}")
    else:
        dest.mkdir(parents=True, exist_ok=True)
        download(dest)
        shutil.make_archive("/content/tmp", "zip", dest)
        shutil.copy("/content/tmp.zip", z)
        print(f"{name}: downloaded and zipped to {z}")
    n = sum(1 for p in dest.rglob("*") if p.suffix.lower() in ds.IMAGE_EXTS)
    print(f"  {n} image files under {dest}")
    return dest

## 3. Roboflow Universe sources (YOLO format)

PH BB v5 reuses the zip that `train_detector_v0.ipynb` saved, if it is on Drive. The class names of each
download are printed so the `plate_classes` ids in `ml/datasets.yaml` can be checked.

In [ ]:
from roboflow import Roboflow

ROBOFLOW = {   # name in ml/datasets.yaml -> (workspace, project, version)
    "license-plate-detection-ph-bb-v5": ("alyssa-marie-jcsyy", "license-plate-detection-ph-bb", 5),
    "usp-ph-plates-v2": ("university-of-southeastern-philippines", "philippine-license-plates", 2),
    "ph-plate-vehicle-v7": ("research-3wj93", "philippine-plate-and-vehicle", 7),
    "motorcycle-plate-zeroexperiments-v5": ("zeroexperiments", "motorcycle-license-plate-skrdr", 5),
    "motorcycle-plate-hanoi-v2": ("hanoi-university-of-industry", "motorcycle-license-plate", 2),
    "keyboard-rover-v4": ("rover-hdxdc", "keyboard-dx28x", 4),
}

def roboflow_download(ws, proj, ver):
    def run(dest):
        key = secret("ROBOFLOW_API_KEY")
        assert key, "Add ROBOFLOW_API_KEY to Colab secrets and enable Notebook access."
        Roboflow(api_key=key).workspace(ws).project(proj).version(ver).download("yolov8", location=str(dest), overwrite=True)
    return run

# The v0 notebook cached PH BB v5 as one zip in MyDrive/CenpAI/data; copy it into raw-zips if present.
old = ROOT / "data" / "license-plate-detection-ph-bb-v5-yolov8.zip"
new = ZIPS / "license-plate-detection-ph-bb-v5.zip"
if old.is_file() and not new.is_file():
    shutil.copy(old, new)

for name, (ws, proj, ver) in ROBOFLOW.items():
    d = fetch(name, roboflow_download(ws, proj, ver))
    y = d / "data.yaml"
    if y.is_file():
        print("  names:", yaml.safe_load(y.read_text()).get("names"))

## 4. PH BB v5: the label files the v0 check rejected

The v0 notebook (and the paper) kept only label files whose every row has 5 fields; 576 were rejected.
This cell finds those files again, says what kind they are, and shows a few with their boxes.
`ml.dataset` turns polygon rows into their bounding box, so polygon files are recovered when the dataset is built;
anything still malformed stays out and is counted as `invalid_label`.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

phbb = src_dir("license-plate-detection-ph-bb-v5")
labels = sorted(p for p in phbb.rglob("*.txt") if p.parent.name == "labels")
rejected = [p for p in labels if any(len(l.split()) != 5 for l in p.read_text().splitlines() if l.strip())]
print("PH BB v5 label files:", len(labels), "| rejected by the 5-field check:", len(rejected))
print("kinds of the rejected files:", dict(ds.inspect_labels(rejected)))
print("kinds of all files:", dict(ds.inspect_labels(labels)))
for p in rejected[:3]:
    print(p.name, "->", p.read_text().splitlines()[0][:120])

def image_for(label):
    for ext in (".jpg", ".jpeg", ".png"):
        q = Path(str(label.parent).replace("/labels", "/images")) / (label.stem + ext)
        if q.is_file():
            return q

def show_boxes(items, title, cols=4):
    rows = max(1, (len(items) + cols - 1) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.2 * rows))
    for ax in np.ravel(axes):
        ax.axis("off")
    for ax, (img_path, boxes) in zip(np.ravel(axes), items):
        im = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = im.shape[:2]
        for cx, cy, bw, bh in boxes:
            cv2.rectangle(im, (int((cx - bw / 2) * w), int((cy - bh / 2) * h)), (int((cx + bw / 2) * w), int((cy + bh / 2) * h)), (255, 0, 0), max(2, w // 200))
        ax.imshow(im)
        ax.set_title(f"{len(boxes)} box(es)", fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

sample = random.Random(0).sample(rejected, min(8, len(rejected)))
items = []
for p in sample:
    try:
        items.append((image_for(p), ds.parse_label(p.read_text(), {0})))
    except ValueError as e:
        print(p.name, "still invalid:", e)
show_boxes([i for i in items if i[0]], "Rejected PH BB v5 labels after polygon -> box conversion (check the boxes fit the plates)")

## 5. Open Images V7, class "Vehicle registration plate"

Downloaded with FiftyOne, keeping Google's official train, validation and test splits (exported as `train`, `val`, `test`).
Images with a "group of" box (one box over several plates) are left out, because a group box does not mark each plate.
Annotations are CC BY 4.0 (Google); images are listed as CC BY 2.0, so the authors are credited:
`open-images-v7-credits.csv` on Drive lists author, license and source URL for every kept image.

In [ ]:
import fiftyone as fo, fiftyone.zoo as foz
from fiftyone import ViewField as F
import pandas as pd

OI_CLASS = "Vehicle registration plate"
OI_SPLITS = {"train": "train", "validation": "val", "test": "test"}   # Google's name -> folder name in ml/datasets.yaml
OI_META = {
    "train": "https://storage.googleapis.com/openimages/2018_04/train/train-images-boxable-with-rotation.csv",
    "validation": "https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv",
    "test": "https://storage.googleapis.com/openimages/2018_04/test/test-images-with-rotation.csv",
}
OI_COUNTS = {}

def open_images(dest):
    kept_ids = []
    for split, ours in OI_SPLITS.items():
        d = foz.load_zoo_dataset("open-images-v7", split=split, label_types=["detections"], classes=[OI_CLASS],
                                 only_matching=True, dataset_name=f"oi-plates-{split}", overwrite=True)
        view = d.match(F("ground_truth.detections").filter(F("IsGroupOf") == True).length() == 0)
        OI_COUNTS[ours] = dict(downloaded=len(d), kept=len(view), dropped_group_of=len(d) - len(view))
        view.export(export_dir=str(dest), dataset_type=fo.types.YOLOv5Dataset, label_field="ground_truth",
                    split=ours, classes=[OI_CLASS])
        kept_ids += [Path(s.filepath).stem for s in view]
    credits = []
    for split, url in OI_META.items():
        m = pd.read_csv(url, usecols=["ImageID", "Subset", "License", "Author", "AuthorProfileURL", "OriginalLandingURL"])
        credits.append(m[m.ImageID.isin(set(kept_ids))])
    pd.concat(credits).to_csv(dest / "open-images-v7-credits.csv", index=False)

oi = fetch("open-images-v7-plates", open_images)
print("counts in this run (empty when restored from Drive):", OI_COUNTS)
shutil.copy(oi / "open-images-v7-credits.csv", OUT_DRIVE / "open-images-v7-credits.csv")
for s in ("train", "val", "test"):
    print(s, "images:", sum(1 for p in (oi / "images" / s).glob("*")))

## 6. CCPD 2019 sample (Blur, Tilt, Rotate, FN and Challenge subsets)

MIT license per the official repository (github.com/detectRecog/CCPD); cite Xu et al., ECCV 2018.
The full archive is large, so only a sample is kept: `CCPD_PER_SUBSET` images from each subset, renamed
`<subset>_<n>.jpg` (CCPD file names encode the plate number; the box is read from the name first).
If `gdown` hits the Google Drive download limit, add the archive to your own Drive from the README link,
set `CCPD_ARCHIVE` to its path, and run the cell again.

In [ ]:
CCPD_PER_SUBSET = 400   #@param {type:"integer"}
CCPD_SUBSETS = ["ccpd_blur", "ccpd_tilt", "ccpd_rotate", "ccpd_fn", "ccpd_challenge"]
CCPD_ARCHIVE = ""       #@param {type:"string"}
CCPD_GDRIVE_ID = "1rdEsCUcIUaYOVRkx5IMTRNA7PcGMmSgc"   # CCPD2019.tar.xz, linked from the official README

def ccpd(dest):
    arch = Path(CCPD_ARCHIVE) if CCPD_ARCHIVE else Path("/content/CCPD2019.tar.xz")
    if not arch.is_file():
        subprocess.run(["gdown", CCPD_GDRIVE_ID, "-O", str(arch)], check=True)
    subprocess.run(["tar", "-xJf", str(arch), "-C", "/content"] + [f"CCPD2019/{s}" for s in CCPD_SUBSETS], check=True)
    rng = random.Random(0)
    for s in CCPD_SUBSETS:
        files = sorted(Path("/content/CCPD2019", s).glob("*.jpg"))
        for i, f in enumerate(rng.sample(files, min(CCPD_PER_SUBSET, len(files)))):
            img = cv2.imread(str(f))
            h, w = img.shape[:2]
            (dest / "images" / s).mkdir(parents=True, exist_ok=True)
            (dest / "labels" / s).mkdir(parents=True, exist_ok=True)
            shutil.copy(f, dest / "images" / s / f"{s}_{i:04d}.jpg")
            (dest / "labels" / s / f"{s}_{i:04d}.txt").write_text(ds.ccpd_label(f.name, w, h))
        print(s, "files in the archive:", len(files))

fetch("ccpd2019-sample", ccpd)

## 7. Look at 20 random images from each source (vetting step 2)

Boxes are drawn as `ml.dataset` will read them (plate classes only). Check that the images match `docs/datasets.md`
and that the boxes sit on plates. Note anything odd before building.

In [ ]:
def sample_items(src, n=20):
    root = ds.resolve_path(src["path"])
    imgs = sorted(p for p in root.rglob("*") if p.suffix.lower() in ds.IMAGE_EXTS)
    items = []
    for p in random.Random(1).sample(imgs, min(n, len(imgs))):
        lbl = ds.label_path(p)
        try:
            boxes = [] if src.get("negatives") or not lbl.is_file() else ds.parse_label(lbl.read_text(), set(src.get("plate_classes", [0])))
        except ValueError:
            boxes = []
        items.append((p, boxes))
    return items

for src in CFG["sources"]:
    if ds.resolve_path(src["path"]).is_dir():
        show_boxes(sample_items(src), src["name"], cols=5)
    else:
        print("not downloaded:", src["name"])

## 8. Keyboard images: find any that show a plate (optional, uses the v0 detector)

Runs detector v0 (`MyDrive/CenpAI/release/yolov8n/best.pt`) on the keyboard images and shows those with a detection at
0.5 or more. v0 is known to mask keyboards, so most hits are keyboards: add to `exclude` in `ml/datasets.yaml`
(path relative to the source folder, as printed) only the images that really show a plate, then rebuild.

In [ ]:
V0 = ROOT / "release" / "yolov8n" / "best.pt"
if V0.is_file():
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "ultralytics"], check=True)
    from ultralytics import YOLO
    kb = src_dir("keyboard-rover-v4")
    imgs = sorted(p for p in kb.rglob("*") if p.suffix.lower() in ds.IMAGE_EXTS)
    hits = []
    for r in YOLO(str(V0)).predict([str(p) for p in imgs], conf=0.5, verbose=False, stream=True):
        if len(r.boxes):
            hits.append((Path(r.path), [tuple(b) for b in r.boxes.xywhn.cpu().numpy()]))
    print(f"{len(hits)} of {len(imgs)} keyboard images have a v0 detection at 0.5 or more")
    for p, _ in hits[:40]:
        print("  ", p.relative_to(kb).as_posix())
    if hits:
        show_boxes(hits[:20], "v0 detections on keyboard images (exclude only real plates)", cols=5)
else:
    print("v0 weights not found at", V0, "- skipped")

## 9. Build dataset v2 and check for leaks

`ml.dataset` groups images by original source name and by near duplicates (dHash 4 or less) across every source,
splits PH BB v5 70/15/15 by group, keeps Open Images' official splits, and keeps every other source in training.
The two printed leak counts must be `0`.

In [ ]:
OUT = WORK / "processed"
if (OUT / CFG["version"]).exists():
    shutil.rmtree(OUT / CFG["version"])      # local copy only; the saved version on Drive is never overwritten
report = ds.prepare(CFG, OUT, seed=0)
print(json.dumps({k: v for k, v in report.items() if k != "sources"}, indent=2))
V2 = OUT / CFG["version"]

In [ ]:
leaks = ds.leak_report(V2)
print(json.dumps(leaks, indent=2))
print("source groups that appear in more than one split:", leaks["groups_in_more_than_one_split"])
print("images shared between a training set and a validation or test split:",
      leaks["eval_images_exactly_in_train"] + leaks["eval_images_near_duplicate_of_train"] + leaks["images_in_both_val_and_test"])
assert leaks["groups_in_more_than_one_split"] == 0
assert leaks["eval_images_exactly_in_train"] + leaks["eval_images_near_duplicate_of_train"] + leaks["images_in_both_val_and_test"] == 0

## 10. Counts per source and the training mix

The protocol keeps internet data to at most about half of the training images, with Philippine plates the larger share.
If the internet share is above half, set `max_images` on the largest internet source in `ml/datasets.yaml`,
bump `version`, and rebuild. Copy this table into `docs/datasets.md`.

In [ ]:
PH = {"license-plate-detection-ph-bb-v5", "usp-ph-plates-v2", "ph-plate-vehicle-v7"}
print("| source | train images | train boxes | val images | val boxes | test images | test boxes | no-plate images |")
print("|---|---|---|---|---|---|---|---|")
for name, c in report["sources"].items():
    neg = sum(c[s]["negatives"] for s in ds.SPLITS)
    print(f"| `{name}` | {c['train']['images']} | {c['train']['boxes']} | {c['val']['images']} | {c['val']['boxes']} | "
          f"{c['test']['images']} | {c['test']['boxes']} | {neg} |")
train_total = report["splits"]["train"]["images"]
ph_train = sum(report["sources"][n]["train"]["images"] for n in PH if n in report["sources"])
print(f"\ntraining images: {train_total}; Philippine sources {ph_train} ({ph_train / train_total:.1%}), "
      f"other sources {train_total - ph_train} ({(train_total - ph_train) / train_total:.1%})")
if train_total - ph_train > ph_train:
    print("Internet data is the larger share of training: cap it with max_images, or oversample PH BB in the training notebook.")

## 11. Save the version to Drive

Zips the built version to `MyDrive/CenpAI/data/processed/dataset-<version>.zip` and copies `manifest.csv` and
`report.json` beside it. The training notebook unzips it to local disk. A saved version is never overwritten.

In [ ]:
zip_path = OUT_DRIVE / f"dataset-{CFG['version']}.zip"
if zip_path.exists():
    print(zip_path, "already exists: bump `version` in ml/datasets.yaml instead of overwriting it")
else:
    shutil.make_archive(f"/content/dataset-{CFG['version']}", "zip", V2)
    shutil.copy(f"/content/dataset-{CFG['version']}.zip", zip_path)
    for f in ("manifest.csv", "report.json"):
        shutil.copy(V2 / f, OUT_DRIVE / f"dataset-{CFG['version']}-{f}")
    h = hashlib.sha256(zip_path.read_bytes()).hexdigest()
    print("saved", zip_path, "sha256", h)